# Notebook 3: Root-Cause Decomposition & Diagnostic Report
**Project:** Retail Site Failure Early-Warning System  
**Input:** `outputs/anomaly_scores.csv` + `data/raw/ga4_sessions.csv`  
**Output:** `outputs/diagnostic_report.html` + `outputs/root_cause_chart.html`

## What this notebook does
1. Identifies critical and warning days from Notebook 2
2. Establishes a baseline CVR from preceding normal days
3. Decomposes each anomaly across 4 dimensions:
   device, country, traffic source, traffic medium
4. Quantifies revenue impact per anomaly day
5. Generates an automated HTML diagnostic report —
   the deliverable a QuantumBlack team hands to a retail client

In [4]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/home/codespace/.config/gcloud/application_default_credentials.json"
)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# Load both datasets
df_anomaly = pd.read_csv('../outputs/anomaly_scores.csv')
df_anomaly['event_date'] = pd.to_datetime(df_anomaly['event_date'])

df_sessions = pd.read_csv('data/raw/ga4_sessions_sample.csv')
df_sessions['event_date'] = pd.to_datetime(df_sessions['event_date'])

# Identify flagged days
critical_days = df_anomaly[
    df_anomaly['anomaly_severity'] == 'critical'
]['event_date'].tolist()

warning_days = df_anomaly[
    df_anomaly['anomaly_severity'] == 'warning'
]['event_date'].tolist()

all_flagged = critical_days + warning_days

print(f"✓ Data loaded")
print(f"  Anomaly scores: {len(df_anomaly)} days")
print(f"  Session data:   {len(df_sessions):,} sessions")
print(f"\nFlagged days:")
print(f"  Critical: {[d.date() for d in critical_days]}")
print(f"  Warning:  {[d.date() for d in warning_days]}")
print(f"  Total:    {len(all_flagged)} days to decompose")

✓ Data loaded
  Anomaly scores: 59 days
  Session data:   5,500 sessions

Flagged days:
  Critical: [datetime.date(2021, 1, 27)]
  Total:    6 days to decompose


In [5]:
def compute_baselines(df_anomaly, all_flagged, window_days=14):
    """
    Baseline CVR = median CVR of normal days in the
    14-day window preceding each flagged date.
    Median used over mean — more robust to other anomalies
    in the window. Standard practice in retail analytics.
    """
    normal_days = df_anomaly[
        df_anomaly['anomaly_severity'] == 'normal'
    ]
    
    baselines = {}
    for flagged_date in all_flagged:
        preceding = normal_days[
            (normal_days['event_date'] < flagged_date) &
            (normal_days['event_date'] >= 
             flagged_date - pd.Timedelta(days=window_days))
        ]
        
        if len(preceding) > 0:
            baselines[flagged_date] = (
                preceding['conversion_rate'].median()
            )
        else:
            baselines[flagged_date] = (
                normal_days['conversion_rate'].median()
            )
    
    return baselines


baselines = compute_baselines(df_anomaly, all_flagged)

print("Baseline CVR vs actual for flagged days:")
print(f"\n{'Date':<14} {'Actual':>8} {'Baseline':>10} "
      f"{'Drop':>8} {'Severity':>10}")
print("-" * 55)

for date in sorted(all_flagged):
    actual = df_anomaly[
        df_anomaly['event_date'] == date
    ]['conversion_rate'].values[0]
    baseline = baselines[date]
    drop = (actual - baseline) / baseline
    severity = df_anomaly[
        df_anomaly['event_date'] == date
    ]['anomaly_severity'].values[0]
    print(f"{str(date.date()):<14} {actual:>8.2%} "
          f"{baseline:>10.2%} {drop:>8.1%} {severity:>10}")

Baseline CVR vs actual for flagged days:

Date             Actual   Baseline     Drop   Severity
-------------------------------------------------------
2020-11-08        1.44%      1.09%    32.7%    warning
2020-12-12        2.41%      1.52%    58.3%    warning
2020-12-19        1.12%      1.09%     2.9%    warning
2020-12-31        0.85%      0.71%    20.2%    warning
2021-01-22        2.17%      0.92%   136.7%    warning
2021-01-27        0.00%      1.11%  -100.0%   critical


In [6]:
def decompose_anomaly(df_sessions, df_anomaly,
                      flagged_date, baseline_cvr, dimension):
    """
    For one anomaly date and one dimension:
    Compare each segment's CVR vs its baseline,
    weighted by its share of traffic.
    
    Contribution = traffic share × CVR deviation
    This correctly weights high-traffic segments —
    a 50% drop in a 5% traffic segment matters less
    than a 10% drop in a 60% traffic segment.
    """
    day_sessions = df_sessions[
        df_sessions['event_date'] == flagged_date
    ]
    if len(day_sessions) == 0:
        return None
    
    normal_dates = df_anomaly[
        (df_anomaly['anomaly_severity'] == 'normal') &
        (df_anomaly['event_date'] < flagged_date) &
        (df_anomaly['event_date'] >=
         flagged_date - pd.Timedelta(days=14))
    ]['event_date']
    
    baseline_sessions = df_sessions[
        df_sessions['event_date'].isin(normal_dates)
    ]
    
    results = []
    for segment in day_sessions[dimension].dropna().unique():
        seg_day = day_sessions[day_sessions[dimension] == segment]
        seg_cvr = seg_day['converted'].mean()
        seg_volume = len(seg_day)
        seg_share = seg_volume / len(day_sessions)
        
        seg_base = baseline_sessions[
            baseline_sessions[dimension] == segment
        ]
        seg_baseline_cvr = (
            seg_base['converted'].mean()
            if len(seg_base) > 0 else baseline_cvr
        )
        
        cvr_drop = seg_cvr - seg_baseline_cvr
        contribution = seg_share * cvr_drop
        
        results.append({
            'segment': str(segment),
            'dimension': dimension,
            'sessions': seg_volume,
            'traffic_share': seg_share,
            'actual_cvr': seg_cvr,
            'baseline_cvr': seg_baseline_cvr,
            'cvr_drop': cvr_drop,
            'cvr_drop_pct': (cvr_drop / seg_baseline_cvr
                             if seg_baseline_cvr > 0 else 0),
            'contribution': contribution
        })
    
    return pd.DataFrame(results).sort_values(
        'contribution'
    ).reset_index(drop=True)


# Run decomposition on critical day
critical_date = critical_days[0]
baseline_cvr = baselines[critical_date]

actual_cvr = df_anomaly[
    df_anomaly['event_date'] == critical_date
]['conversion_rate'].values[0]

total_sessions = df_anomaly[
    df_anomaly['event_date'] == critical_date
]['total_sessions'].values[0]

total_revenue = df_anomaly[
    df_anomaly['event_date'] == critical_date
]['total_revenue'].values[0]

# $50 assumed AOV for Google Merchandise Store
revenue_impact = (
    (actual_cvr - baseline_cvr) * total_sessions * 50
)

print(f"{'='*52}")
print(f"CRITICAL ANOMALY: {critical_date.date()}")
print(f"{'='*52}")
print(f"  Actual CVR:          {actual_cvr:.2%}")
print(f"  Baseline CVR:        {baseline_cvr:.2%}")
print(f"  Drop:                "
      f"{(actual_cvr-baseline_cvr)/baseline_cvr:.1%}")
print(f"  Sessions affected:   {total_sessions:,}")
print(f"  Est. revenue impact: ${abs(revenue_impact):,.0f}")
print(f"{'='*52}\n")

dimensions = [
    'device_category', 'country',
    'traffic_source', 'traffic_medium'
]

decompositions = {}
for dim in dimensions:
    result = decompose_anomaly(
        df_sessions, df_anomaly,
        critical_date, baseline_cvr, dim
    )
    if result is not None and len(result) > 0:
        decompositions[dim] = result
        print(f"\n--- {dim.upper().replace('_',' ')} ---")
        display(result[[
            'segment', 'traffic_share', 'actual_cvr',
            'baseline_cvr', 'cvr_drop_pct', 'contribution'
        ]].round(4))

CRITICAL ANOMALY: 2021-01-27
  Actual CVR:          0.00%
  Baseline CVR:        1.11%
  Drop:                -100.0%
  Sessions affected:   167
  Est. revenue impact: $93


--- DEVICE CATEGORY ---


,segment,traffic_share,actual_cvr,baseline_cvr,cvr_drop_pct,contribution
0,desktop,0.6170,0.0345,0.0526,-0.3448,-0.0112
1,tablet,0.0106,0.0000,0.0909,-1.0000,-0.0010
2,mobile,0.3723,0.2286,0.1048,1.1810,0.0461



--- COUNTRY ---


,segment,traffic_share,actual_cvr,baseline_cvr,cvr_drop_pct,contribution
0,Canada,0.0426,0.0000,0.2093,-1.0000,-0.0089
1,Taiwan,0.0213,0.0000,0.2353,-1.0000,-0.0050
2,United States,0.5000,0.0638,0.0711,-0.1024,-0.0036
3,Ukraine,0.0106,0.0000,0.3333,-1.0000,-0.0035
4,Singapore,0.0213,0.0000,0.0909,-1.0000,-0.0019
5,France,0.0106,0.0000,0.1333,-1.0000,-0.0014
6,Netherlands,0.0106,0.0000,0.1111,-1.0000,-0.0012
7,China,0.0106,0.0000,0.1111,-1.0000,-0.0012
8,South Korea,0.0213,0.0000,0.0000,0.0000,0.0000
9,Brazil,0.0106,0.0000,0.0000,0.0000,0.0000



--- TRAFFIC SOURCE ---


,segment,traffic_share,actual_cvr,baseline_cvr,cvr_drop_pct,contribution
0,google,0.3936,0.0541,0.0895,-0.3959,-0.0139
1,<Other>,0.2447,0.0435,0.0455,-0.0435,-0.0005
2,(data deleted),0.0745,0.2857,0.1852,0.5429,0.0075
3,shop.googlemerchandisestore.com,0.0851,0.2500,0.1333,0.8750,0.0099
4,(direct),0.2021,0.1579,0.0469,2.3684,0.0224



--- TRAFFIC MEDIUM ---


,segment,traffic_share,actual_cvr,baseline_cvr,cvr_drop_pct,contribution
0,cpc,0.0638,0.0000,0.1481,-1.0000,-0.0095
1,organic,0.3830,0.0556,0.0773,-0.2817,-0.0083
2,referral,0.1809,0.1176,0.1089,0.0802,0.0016
3,(data deleted),0.0745,0.2857,0.1852,0.5429,0.0075
4,<Other>,0.0957,0.1111,0.0125,7.8889,0.0094
5,(none),0.2021,0.1579,0.0469,2.3684,0.0224


In [7]:
# Root cause visualization — 4-panel decomposition chart
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        d.replace('_', ' ').title()
        for d in decompositions.keys()
    ],
    vertical_spacing=0.22,
    horizontal_spacing=0.14
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for idx, (dim, df_decomp) in enumerate(decompositions.items()):
    if idx >= 4:
        break
    row, col = positions[idx]
    df_plot = df_decomp.nsmallest(5, 'contribution')
    colors = [
        '#E53935' if c < 0 else '#43A047'
        for c in df_plot['contribution']
    ]
    
    fig.add_trace(
        go.Bar(
            x=df_plot['segment'].astype(str),
            y=df_plot['cvr_drop_pct'],
            marker_color=colors,
            text=[f"{v:.1%}" for v in df_plot['cvr_drop_pct']],
            textposition='outside',
            showlegend=False,
            hovertemplate=(
                '%{x}<br>CVR drop: %{y:.2%}<extra></extra>'
            )
        ),
        row=row, col=col
    )

fig.update_layout(
    title=dict(
        text=(
            f"<b>Root-cause decomposition: "
            f"{critical_date.date()} critical anomaly</b><br>"
            f"<sup>CVR dropped from {baseline_cvr:.2%} to "
            f"{actual_cvr:.2%} — "
            f"est. ${abs(revenue_impact):,.0f} revenue impact · "
            f"Red = below baseline</sup>"
        ),
        font=dict(size=13)
    ),
    height=580,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font_family='Arial'
)
fig.update_yaxes(tickformat='.1%')
fig.update_xaxes(tickangle=-20)
fig.add_annotation(
    text=(
        "Source: Google Merchandise Store GA4 · "
        "Method: Segment contribution = "
        "traffic share × CVR deviation from 14-day baseline"
    ),
    xref="paper", yref="paper",
    x=0, y=-0.12, showarrow=False,
    font=dict(size=9, color='grey')
)

fig.show()
fig.write_html('../outputs/root_cause_chart.html')
print("✓ Root cause chart saved")

✓ Root cause chart saved


In [8]:
def generate_diagnostic_report(
    critical_date, actual_cvr, baseline_cvr,
    revenue_impact, decompositions, total_sessions
):
    top_findings = []
    for dim, df_d in decompositions.items():
        if len(df_d) > 0:
            worst = df_d.iloc[0]
            if worst['cvr_drop_pct'] < -0.05:
                top_findings.append({
                    'dimension': dim.replace('_',' ').title(),
                    'segment': worst['segment'],
                    'drop': worst['cvr_drop_pct'],
                    'share': worst['traffic_share']
                })
    
    findings_html = "".join([f"""
    <div class="finding">
        <span class="finding-dim">{f['dimension']}</span>
        <span class="finding-seg">{f['segment']}</span>
        CVR dropped
        <span class="finding-drop">{f['drop']:.1%}</span>
        vs baseline ({f['share']:.0%} of traffic)
    </div>""" for f in top_findings]) or (
        "<p>No single segment accounts for >5% of the drop "
        "— likely a site-wide issue.</p>"
    )
    
    html = f"""<!DOCTYPE html>
<html>
<head>
<style>
  body{{font-family:Arial,sans-serif;max-width:820px;
       margin:40px auto;color:#1a1a1a;line-height:1.6}}
  .header{{background:#1a1a2e;color:white;padding:24px 32px;
           border-radius:8px;margin-bottom:24px}}
  .header h1{{margin:0 0 4px 0;font-size:20px}}
  .header .sub{{opacity:.7;font-size:13px}}
  .alert{{background:#ffebee;border-left:4px solid #E53935;
          padding:16px 20px;border-radius:4px;margin-bottom:20px}}
  .alert .label{{font-size:11px;font-weight:600;color:#E53935;
                 text-transform:uppercase;letter-spacing:.05em}}
  .alert .value{{font-size:26px;font-weight:700;
                 color:#E53935;margin:4px 0}}
  .metrics{{display:grid;grid-template-columns:1fr 1fr 1fr;
            gap:16px;margin-bottom:24px}}
  .metric{{background:#f8f9fa;padding:16px;
           border-radius:6px;text-align:center}}
  .metric .m-label{{font-size:11px;color:#666;
                    text-transform:uppercase}}
  .metric .m-value{{font-size:22px;font-weight:600;margin-top:4px}}
  .section{{margin-bottom:24px}}
  .section h2{{font-size:15px;border-bottom:1px solid #eee;
               padding-bottom:8px;margin-bottom:12px}}
  .finding{{background:#fff3e0;border-left:3px solid #FF6F00;
            padding:10px 14px;margin-bottom:8px;
            border-radius:4px;font-size:13px}}
  .finding-dim{{font-size:10px;font-weight:600;color:#FF6F00;
                text-transform:uppercase;margin-right:8px}}
  .finding-seg{{font-weight:600;margin-right:6px}}
  .finding-drop{{color:#E53935;font-weight:600}}
  .rec{{background:#e8f5e9;border-left:3px solid #43A047;
        padding:10px 14px;margin-bottom:8px;
        border-radius:4px;font-size:13px}}
  .footer{{font-size:11px;color:#999;margin-top:32px;
           padding-top:16px;border-top:1px solid #eee}}
</style>
</head>
<body>
<div class="header">
  <h1>⚠ Site Conversion Anomaly — Diagnostic Report</h1>
  <div class="sub">Auto-generated · Date: {critical_date.date()} 
  · Severity: CRITICAL</div>
</div>
<div class="alert">
  <div class="label">Conversion Rate Drop Detected</div>
  <div class="value">
    {actual_cvr:.2%} vs {baseline_cvr:.2%} baseline
    ({(actual_cvr-baseline_cvr)/baseline_cvr:.1%})
  </div>
  Est. revenue impact: <strong>${abs(revenue_impact):,.0f}</strong>
  on {total_sessions:,} sessions
</div>
<div class="metrics">
  <div class="metric">
    <div class="m-label">Actual CVR</div>
    <div class="m-value" style="color:#E53935">{actual_cvr:.2%}</div>
  </div>
  <div class="metric">
    <div class="m-label">Baseline CVR</div>
    <div class="m-value">{baseline_cvr:.2%}</div>
  </div>
  <div class="metric">
    <div class="m-label">Sessions Affected</div>
    <div class="m-value">{total_sessions:,}</div>
  </div>
</div>
<div class="section">
  <h2>Root-Cause Findings</h2>
  {findings_html}
</div>
<div class="section">
  <h2>Recommended Actions</h2>
  <div class="rec"><strong>Immediate (0–2 hrs):</strong>
  Check deployment logs for changes in the 24 hours preceding
  {critical_date.date()} — especially checkout or payment updates.
  </div>
  <div class="rec"><strong>Short-term (2–24 hrs):</strong>
  Review affected segments above for device or geo-specific issues.
  Cross-reference with CDN and server error logs.</div>
  <div class="rec"><strong>Follow-up (24–72 hrs):</strong>
  Implement segment-level monitoring so future drops in any single
  dimension trigger alerts before affecting the overall rate.</div>
</div>
<div class="footer">
  Early-Warning System · Retail Site Failure Detection ·
  Isolation Forest (contamination=0.05) + CUSUM (Pelt, RBF) ·
  Google Merchandise Store GA4 Public Dataset
</div>
</body>
</html>"""
    
    return html


report_html = generate_diagnostic_report(
    critical_date, actual_cvr, baseline_cvr,
    revenue_impact, decompositions, total_sessions
)

with open('../outputs/diagnostic_report.html', 'w') as f:
    f.write(report_html)

print("✓ Diagnostic report saved to outputs/diagnostic_report.html")
display(HTML(report_html))

✓ Diagnostic report saved to outputs/diagnostic_report.html


## Notebook 3 Complete — Project Complete

**What was built across 3 notebooks:**
1. `01_data_pipeline.ipynb` — GA4 BigQuery ingestion, 
   222k sessions, session feature engineering
2. `02_anomaly_detection.ipynb` — Isolation Forest + CUSUM,
   structural breaks identified, severity scoring
3. `03_root_cause_decomposition.ipynb` — segment contribution
   decomposition, revenue impact quantification, 
   automated diagnostic report

**All output files:**
- `outputs/anomaly_scores.csv`
- `outputs/anomaly_detection_chart.html`
- `outputs/root_cause_chart.html`
- `outputs/diagnostic_report.html`

**What this system does in production:**
Notebooks 2 and 3 run on a daily schedule. When anomaly score
exceeds threshold, the diagnostic report generates automatically
and is sent to the retail operations team — reducing mean time
to diagnosis from hours to minutes.